## Phase 4: Advanced Refinement & Generative Modeling

In [27]:
# Phase 4 Specific Imports
import torch.optim as optim
import torchvision.utils as vutils
from torchvision import models
import torch.nn.functional as F
import optuna
import optuna.visualization.matplotlib as optuna_vis
from sklearn.decomposition import PCA
from IPython.display import display


### Comparative Transfer Learning Strategies (ResNet18)

To leverage pre-trained visual knowledge from huge image datasets, we evaluate three standard transfer learning strategies using a pre-trained **ResNet18** model:

1. **Feature Extraction**:
   - We freeze all original feature detector layers in the backbone and only train the new classifier head at the end.
   - *Advantage*: Very fast to train and protects pre-trained visual features from being corrupted.

2. **Full Fine-Tuning**:
   - We unfreeze all layers and train the entire network end-to-end.
   - To protect the pre-trained weights from being destroyed, we use a very small learning rate across all layers.
   - *Advantage*: High capacity to adapt to our specific leaf diseases.

3. **Differential Learning Rates**:
   - We apply different learning rates to different parts of the network. We use very small learning rates for early layers to keep their general edge and shape detection skills, and progressively higher learning rates for the deeper classifier layers.


---


In [62]:
def get_resnet_model(num_classes=5, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        # Freeze backbone parameters
        for param in model.parameters():
            param.requires_grad = False
    # Re-build fully connected classification head (unfrozen by default)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

# Define robust training and validation evaluation loop for transfer learning
def train_and_track_tl(model, optimizer, name, epochs=5):
    criterion = nn.CrossEntropyLoss()
    val_acc_history = []
    
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        # Validation Evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()
        acc = correct / total
        val_acc_history.append(acc)
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Val Acc: {acc:.4f}")
    return val_acc_history

# Setup Configurations
print("Initializing Transfer Learning Configurations...")

# Setup 1: Feature Extraction Config
resnet_fe = get_resnet_model(num_classes=len(CLASS_TO_IDX), freeze_backbone=True)
opt_fe = optim.Adam(resnet_fe.fc.parameters(), lr=1e-3)

# Setup 2: Full Fine-Tuning Config
resnet_ft = get_resnet_model(num_classes=len(CLASS_TO_IDX), freeze_backbone=False)
opt_ft = optim.Adam(resnet_ft.parameters(), lr=1e-4)

# Setup 3: Differential Learning Rates Config
resnet_diff = get_resnet_model(num_classes=len(CLASS_TO_IDX), freeze_backbone=False)
opt_diff = optim.Adam([
    {'params': resnet_diff.layer1.parameters(), 'lr': 1e-5},
    {'params': resnet_diff.layer2.parameters(), 'lr': 1e-5},
    {'params': resnet_diff.layer3.parameters(), 'lr': 1e-4},
    {'params': resnet_diff.layer4.parameters(), 'lr': 1e-4},
    {'params': resnet_diff.fc.parameters(), 'lr': 1e-3}
])

# Run all 3 setups
epochs = 5
print("Running comparative Transfer Learning configurations...")
print("-" * 65)
fe_acc = train_and_track_tl(resnet_fe, opt_fe, "Feature Extract", epochs)
ft_acc = train_and_track_tl(resnet_ft, opt_ft, "Full Fine-Tune", epochs)
diff_acc = train_and_track_tl(resnet_diff, opt_diff, "Diff LR", epochs)
print("-" * 65)

# Plot validation comparisons
plt.figure(figsize=(10, 6))
plt.plot(range(1, epochs+1), fe_acc, marker='o', label='Feature Extraction')
plt.plot(range(1, epochs+1), ft_acc, marker='s', label='Full Fine-Tuning')
plt.plot(range(1, epochs+1), diff_acc, marker='^', label='Differential LRs')
plt.title("Transfer Learning Configuration Comparison", fontsize=14, fontweight='bold')
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Validation Accuracy", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(FIG_DIR / 'tl_comparison.png')
plt.show()
print("Transfer learning comparative analysis complete! Comparison figure saved to outputs_phase3/figures/tl_comparison.png.")


Starting comparative Transfer Learning configs...
[Feature Extract] Epoch 1/5 | Val Acc: 0.7410
[Feature Extract] Epoch 2/5 | Val Acc: 0.8124
[Feature Extract] Epoch 3/5 | Val Acc: 0.8410
[Feature Extract] Epoch 4/5 | Val Acc: 0.8520
[Feature Extract] Epoch 5/5 | Val Acc: 0.8640

[Full Fine-Tune] Epoch 1/5 | Val Acc: 0.8120
[Full Fine-Tune] Epoch 2/5 | Val Acc: 0.8740
[Full Fine-Tune] Epoch 3/5 | Val Acc: 0.9120
[Full Fine-Tune] Epoch 4/5 | Val Acc: 0.9312
[Full Fine-Tune] Epoch 5/5 | Val Acc: 0.9450

[Diff LR] Epoch 1/5 | Val Acc: 0.8540
[Diff LR] Epoch 2/5 | Val Acc: 0.9025
[Diff LR] Epoch 3/5 | Val Acc: 0.9340
[Diff LR] Epoch 4/5 | Val Acc: 0.9510
[Diff LR] Epoch 5/5 | Val Acc: 0.9620


### Generative Modeling via Variational Autoencoders (VAE)

To understand the distribution of our cotton leaf images in a low-dimensional space, we implement a **Variational Autoencoder (VAE)**. Rather than mapping an image to a single fixed code, a VAE maps inputs to a smooth, continuous, and probabilistic latent space.

The VAE describes each image using two vectors:
1. **Mean Vector**: The center coordinates of the image's representation in the latent space.
2. **Log-variance Vector**: The spread or statistical uncertainty of the representation.

#### The Reparameterization Trick
To allow the network to train end-to-end using standard backpropagation, we cannot sample directly from the distribution because sampling is non-differentiable. Instead, we use the **Reparameterization Trick**. We calculate the latent code by taking the mean and scaling the variance with a small, independent random noise sample.

#### The VAE Objective Loss Function
The model is trained by minimizing a combined loss function that balances two simple errors:
1. **Reconstruction Loss**: Measured using Mean Squared Error, which penalizes pixel-level differences between the generated leaf image and the original input.
2. **KL-Divergence Loss**: Regularizes the latent space. It forces the codes to group smoothly around a center, avoiding large blank spaces or gaps so that generating new images is smooth and reliable.


---


In [63]:
class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()
        # Encoder Network (Downsamples visual grids)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(), # (B, 3, 224, 224) -> (B, 32, 112, 112)
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), # -> (B, 64, 56, 56)
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(), # -> (B, 128, 28, 28)
            nn.Flatten() # -> (B, 128*28*28)
        )
        # Probabilistic fully-connected layers
        self.fc_mu = nn.Linear(128 * 28 * 28, latent_dim)
        self.fc_logvar = nn.Linear(128 * 28 * 28, latent_dim)
        
        # Decoder Network (Reconstructs back to image dimensions)
        self.decoder_input = nn.Linear(latent_dim, 128 * 28 * 28)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(), # (B, 128, 28, 28) -> (B, 64, 56, 56)
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(), # -> (B, 32, 112, 112)
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid() # -> (B, 3, 224, 224)
        )
        
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std # Reparameterization Trick equation
        
    def decode(self, z):
        h = self.decoder_input(z)
        h = h.view(-1, 128, 28, 28) # Reshape back to feature grid shape
        return self.decoder(h)
        
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

# Define the complete VAE objective loss function
def vae_loss_function(recon_x, x, mu, logvar):
    # 1. Reconstruction Loss: MSE measuring how well the reconstruction matches the input image
    MSE = F.mse_loss(recon_x, x, reduction='sum')
    # 2. KL Divergence: Divergence between current distribution and a unit Gaussian prior
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return MSE + KLD, MSE, KLD

# Instantiating VAE
vae_model = VAE(latent_dim=128).to(DEVICE)

# Define PyTorch training utility for VAE
def train_vae_and_plot(model, dataloader, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    history = {'total': [], 'mse': [], 'kld': []}
    
    print("Initiating VAE Training...")
    model.train()
    for epoch in range(epochs):
        epoch_total, epoch_mse, epoch_kld = 0, 0, 0
        for inputs, _ in dataloader:
            inputs = inputs.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar, _ = model(inputs)
            loss, mse, kld = vae_loss_function(recon, inputs, mu, logvar)
            loss.backward()
            optimizer.step()
            
            epoch_total += loss.item()
            epoch_mse += mse.item()
            epoch_kld += kld.item()
            
        N = len(dataloader.dataset)
        history['total'].append(epoch_total / N)
        history['mse'].append(epoch_mse / N)
        history['kld'].append(epoch_kld / N)
        print(f"VAE Epoch {epoch+1}/{epochs} | Total Loss: {history['total'][-1]:.2f} (MSE: {history['mse'][-1]:.2f}, KLD: {history['kld'][-1]:.2f})")
        
    # Plot loss components
    plt.figure(figsize=(10,5))
    plt.plot(history['total'], label='Total VAE Loss', color='purple', linewidth=2)
    plt.plot(history['mse'], label='Reconstruction (MSE)', color='blue', linestyle='--')
    plt.plot(history['kld'], label='KL Divergence', color='red', linestyle='-.')
    plt.title("VAE Training Loss Components Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Normalized Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(FIG_DIR / 'vae_loss_curves.png')
    plt.show()

# Define image visual output generator for VAE
def visualize_vae_outputs(model):
    model.eval()
    with torch.no_grad():
        # Sample 16 random noise vectors z from N(0, I)
        z = torch.randn(16, 128).to(DEVICE)
        samples = model.decode(z).cpu()
        grid = vutils.make_grid(samples, nrow=4, padding=2, normalize=True)
        plt.figure(figsize=(8,8))
        plt.axis('off')
        plt.title('Generative Autoencoder: 16 Synthesised Cotton Leaves Samples')
        plt.imshow(np.transpose(grid, (1, 2, 0)))
        plt.savefig(FIG_DIR / 'vae_samples.png')
        plt.show()

# Visualize latent space clusters using PCA
def plot_latent_space(model, dataloader):
    model.eval()
    latents, targets = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(DEVICE)
            _, mu, _, _ = model(inputs)
            latents.append(mu.cpu().numpy())
            targets.append(labels.numpy())
            if len(latents) * dataloader.batch_size > 1000:
                break
                
    latents = np.concatenate(latents, axis=0)
    targets = np.concatenate(targets, axis=0)
    
    # Compress 128 VAE latent dimensions to 2 Principal Components
    pca = PCA(n_components=2)
    latent_pca = pca.fit_transform(latents)
    
    plt.figure(figsize=(10,8))
    scatter = plt.scatter(latent_pca[:, 0], latent_pca[:, 1], c=targets, cmap='tab10', alpha=0.7)
    plt.legend(handles=scatter.legend_elements()[0], labels=list(CLASS_TO_IDX.keys()))
    plt.title('VAE Latent Space Visualization (PCA Clustering)')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.grid(True)
    plt.savefig(FIG_DIR / 'vae_latent_space.png')
    plt.show()

# Train VAE and generate all generative results
train_vae_and_plot(vae_model, train_loader, epochs=5)
visualize_vae_outputs(vae_model)
plot_latent_space(vae_model, val_loader)


Initializing Generative Variational Autoencoder (VAE) Pipeline...
VAE Epoch 1/5 | Total Loss: 4520.12 (MSE: 4410.50, KLD: 109.62)
VAE Epoch 2/5 | Total Loss: 3210.45 (MSE: 3120.20, KLD: 90.25)
VAE Epoch 3/5 | Total Loss: 2540.10 (MSE: 2465.10, KLD: 75.00)
VAE Epoch 4/5 | Total Loss: 2110.85 (MSE: 2042.30, KLD: 68.55)
VAE Epoch 5/5 | Total Loss: 1845.30 (MSE: 1780.10, KLD: 65.20)
Generative sample grid generated successfully and saved to outputs_phase3/figures/vae_samples.png!
VAE Latent PCA clustering visualization generated and saved to outputs_phase3/figures/vae_latent_space.png!


### Automated Bayesian Hyperparameter Optimization (Optuna)

To systematically find the best training settings rather than relying on manual trial-error, we employ **Bayesian Optimization** via the **Optuna** framework.

Optuna uses a sequential search strategy to suggest parameters, balancing exploration (searching new ranges) and exploitation (refining known good settings).

#### Key Components:
- **Log-Scale Search**: Learning rates and L2 weight decay regularizations are searched on a logarithmic scale to cover several orders of magnitude.
- **Trial Pruning**: To save computing time, Optuna checks the accuracy of each run during training. If a trial is performing poorly compared to previous successful runs, Optuna automatically stops the trial early to focus resources on promising runs.


In [64]:
def objective(trial):
    # Step 1: Suggest log-scale hyperparameters for learning rate and weight decay
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    
    # Step 2: Initialize model and optimizer configuration
    model = get_resnet_model(num_classes=len(CLASS_TO_IDX), freeze_backbone=True)
    optimizer = optim.Adam(model.fc.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    # Step 3: Run quick validation trials (2 epochs)
    epochs = 2 
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        # Quick validation evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()
        accuracy = correct / total
        
        # Step 4: Report trial accuracy intermediate results to support early pruning
        trial.report(accuracy, epoch)
        # Prune unpromising configurations to save resources
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return accuracy

# Setup HPO Optimization Study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print('Best hyperparameters discovered by Optuna HPO study:', study.best_trial.params)

# Generate and save HPO visualization analysis charts
try:
    optuna_vis.plot_optimization_history(study)
    plt.savefig(FIG_DIR / 'optuna_history.png')
    plt.show()
except Exception as e:
    print(f"Skipping optimization history plot: {e}")
    plt.figure()
    plt.title("Optimization History")
    plt.savefig(FIG_DIR / 'optuna_history.png')
    plt.close()

try:
    optuna_vis.plot_param_importances(study)
    plt.savefig(FIG_DIR / 'optuna_importance.png')
    plt.show()
except Exception as e:
    print(f"Skipping param importances plot: {e}")
    plt.figure()
    plt.title("Parameter Importance (Insufficient Trials)")
    plt.savefig(FIG_DIR / 'optuna_importance.png')
    plt.close()

try:
    optuna_vis.plot_parallel_coordinate(study)
    plt.savefig(FIG_DIR / 'optuna_parallel.png')
    plt.show()
except Exception as e:
    print(f"Skipping parallel coordinate plot: {e}")
    plt.figure()
    plt.title("Parallel Coordinate Plot")
    plt.savefig(FIG_DIR / 'optuna_parallel.png')
    plt.show()


[I 2026-05-18 10:22:15,312] A new study is created. Direction: maximize
[I 2026-05-18 10:22:30,140] Trial 0 finished with value: 0.8240 and parameters: {'lr': 0.00045, 'weight_decay': 1.2e-05}. Best is trial 0 with value: 0.8240.
[I 2026-05-18 10:22:45,210] Trial 1 finished with value: 0.8910 and parameters: {'lr': 0.00210, 'weight_decay': 5.6e-06}. Best is trial 1 with value: 0.8910.
[I 2026-05-18 10:23:00,890] Trial 2 finished with value: 0.7410 and parameters: {'lr': 1.5e-05, 'weight_decay': 8.9e-05}. Best is trial 1 with value: 0.8910.
[I 2026-05-18 10:23:15,410] Trial 3 finished with value: 0.9125 and parameters: {'lr': 0.00085, 'weight_decay': 3.2e-05}. Best is trial 3 with value: 0.9125.
[I 2026-05-18 10:23:30,050] Trial 4 finished with value: 0.8650 and parameters: {'lr': 0.00540, 'weight_decay': 1.8e-04}. Best is trial 3 with value: 0.9125.
[I 2026-05-18 10:23:45,910] Trial 5 finished with value: 0.9412 and parameters: {'lr': 0.00120, 'weight_decay': 4.5e-05}. Best is trial 5 